# Eventbrite Event Pull — Chicago

This notebook pulls events from the Eventbrite API for the Chicago area, including **full descriptions**, categories, and subcategories. These fields are what make Eventbrite more useful than Ticketmaster for this pipeline.

### Setup
1. Go to https://www.eventbrite.com/platform/api and create an account
2. Create a new app and copy your **Private Token**
3. Paste it into the `API_TOKEN` cell below

In [5]:
import requests
import pandas as pd
import json
import time
from datetime import datetime, timezone, timedelta

# -------------------------
# CONFIG — fill in your token
# -------------------------
API_TOKEN  = "VYLXCRCCNS6ZNM6H76AD"   # paste your Private Token here

HEADERS    = {"Authorization": f"Bearer {API_TOKEN}"}

# Chicago lat/long
LAT        = 41.8781
LNG        = -87.6298
RADIUS     = "10km"
MAX_PAGES  = 10
OUTPUT_CSV  = "chicago_eventbrite_events.csv"
OUTPUT_JSON = "chicago_eventbrite_events.json"

DATE_START = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
DATE_END   = (datetime.now(timezone.utc) + timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ")

BASE_URL   = "https://www.eventbriteapi.com/v3/events/search/"

print(f"Pulling events from {DATE_START} to {DATE_END}")

Pulling events from 2026-02-22T21:32:28Z to 2026-03-24T21:32:28Z


# -------------------------
# Connectivity test — run this first to confirm your token works
# -------------------------
test_resp = requests.get(
    "https://www.eventbriteapi.com/v3/users/me/",
    headers=HEADERS
)
print(f"Status: {test_resp.status_code}")
print(test_resp.json())

In [6]:
test_resp = requests.get(
    "https://www.eventbriteapi.com/v3/users/me/",
    headers=HEADERS
)
print(f"Status: {test_resp.status_code}")
print(test_resp.json())

Status: 200
{'emails': [{'email': 'simronpatel2025@u.northwestern.edu', 'verified': True, 'primary': True}], 'id': '2980119458431', 'name': 'Simron Patel', 'first_name': 'Simron', 'last_name': 'Patel', 'is_public': False, 'image_id': None}


In [7]:
def pull_events():
    all_events = []
    continuation = None
    page = 1

    while page <= MAX_PAGES:
        params = {
            "location.latitude":         LAT,
            "location.longitude":        LNG,
            "location.within":           RADIUS,
            "start_date.range_start":    DATE_START,
            "start_date.range_end":      DATE_END,
            "expand":                    "venue,category,subcategory,format",
            "page_size":                 50,
        }
        if continuation:
            params["continuation"] = continuation

        resp = requests.get(BASE_URL, headers=HEADERS, params=params)

        if resp.status_code == 404:
            print("404 — /events/search/ is restricted for this account tier.")
            print("Your token is valid but Eventbrite limits public search for new apps.")
            break

        if resp.status_code != 200:
            print(f"Error on page {page}: {resp.status_code} — {resp.text}")
            break

        data = resp.json()
        events = data.get("events", [])
        all_events.extend(events)
        print(f"Page {page}: fetched {len(events)} events (total so far: {len(all_events)})")

        pagination = data.get("pagination", {})
        if not pagination.get("has_more_items", False):
            print("No more pages.")
            break

        continuation = pagination.get("continuation")
        page += 1
        time.sleep(0.5)

    return all_events

raw_events = pull_events()
print(f"\nTotal raw events fetched: {len(raw_events)}")

404 — /events/search/ is restricted for this account tier.
Your token is valid but Eventbrite limits public search for new apps.

Total raw events fetched: 0


In [2]:
def pull_events():
    all_events = []
    continuation = None
    page = 1

    while page <= MAX_PAGES:
        params = {
            "location.address":          LOCATION,
            "location.within":           RADIUS,
            "start_date.range_start":    DATE_START,
            "start_date.range_end":      DATE_END,
            "expand":                    "venue,category,subcategory,format",
            "page_size":                 50,
        }
        if continuation:
            params["continuation"] = continuation

        resp = requests.get(BASE_URL, headers=HEADERS, params=params)

        if resp.status_code != 200:
            print(f"Error on page {page}: {resp.status_code} — {resp.text}")
            break

        data = resp.json()
        events = data.get("events", [])
        all_events.extend(events)
        print(f"Page {page}: fetched {len(events)} events (total so far: {len(all_events)})")

        pagination = data.get("pagination", {})
        if not pagination.get("has_more_items", False):
            print("No more pages.")
            break

        continuation = pagination.get("continuation")
        page += 1
        time.sleep(0.5)   # be polite to the API

    return all_events

raw_events = pull_events()
print(f"\nTotal raw events fetched: {len(raw_events)}")

Error on page 1: 404 — {"status_code":404,"error_description":"The path you requested does not exist.","error":"NOT_FOUND"}

Total raw events fetched: 0


## Parse into a Flat DataFrame

Extract the fields most useful for the recommendation pipeline:
- `description` — the key field missing from Ticketmaster
- `category` / `subcategory` — Eventbrite's own taxonomy
- `name`, `url`, `start`, `venue` — standard metadata

In [ ]:
def parse_event(e):
    venue    = e.get("venue") or {}
    address  = venue.get("address") or {}
    category = e.get("category") or {}
    subcat   = e.get("subcategory") or {}
    fmt      = e.get("format") or {}

    return {
        "id":               e.get("id", ""),
        "name":             e.get("name", {}).get("text", ""),
        "description":      e.get("description", {}).get("text", ""),
        "url":              e.get("url", ""),
        "start_date":       e.get("start", {}).get("local", ""),
        "end_date":         e.get("end",   {}).get("local", ""),
        "is_free":          e.get("is_free", False),
        "category":         category.get("name", ""),
        "subcategory":      subcat.get("name", ""),
        "format":           fmt.get("name", ""),
        "venue_name":       venue.get("name", ""),
        "venue_address":    address.get("localized_address_display", ""),
        "venue_city":       address.get("city", ""),
        "venue_state":      address.get("region", ""),
        "venue_lat":        venue.get("latitude", ""),
        "venue_long":       venue.get("longitude", ""),
        "capacity":         e.get("capacity", ""),
        "status":           e.get("status", ""),
    }

parsed = [parse_event(e) for e in raw_events]
df = pd.DataFrame(parsed)

# Drop duplicates
df = df.drop_duplicates(subset=["id"])

print(f"Parsed events: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Inspect Description Quality

Check how many events actually have descriptions — this is the key signal for the classifier.

In [ ]:
has_description = df["description"].str.strip().str.len() > 20
has_category    = df["category"].str.strip().str.len() > 0
has_subcategory = df["subcategory"].str.strip().str.len() > 0

print(f"Events with description:  {has_description.sum()} / {len(df)} ({100*has_description.mean():.1f}%)")
print(f"Events with category:     {has_category.sum()} / {len(df)} ({100*has_category.mean():.1f}%)")
print(f"Events with subcategory:  {has_subcategory.sum()} / {len(df)} ({100*has_subcategory.mean():.1f}%)")

print("\n--- Category breakdown ---")
print(df["category"].value_counts())

print("\n--- Subcategory breakdown ---")
print(df["subcategory"].value_counts().head(20))

## Sample Descriptions

Preview a few events with descriptions to judge quality.

In [ ]:
sample = df[has_description][["name", "category", "subcategory", "description"]].head(5)
for _, row in sample.iterrows():
    print(f"NAME:        {row['name']}")
    print(f"CATEGORY:    {row['category']} > {row['subcategory']}")
    print(f"DESCRIPTION: {row['description'][:300]}")
    print("-" * 80)

## Save

Save the full dataset to CSV and JSON for use in the recommendation pipeline.

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
df.to_json(OUTPUT_JSON, orient="records", indent=2)

print(f"Saved {len(df)} events to:")
print(f"  {OUTPUT_CSV}")
print(f"  {OUTPUT_JSON}")